### Experiment Tracking with Weights & Biases

1. Set up a Weights & Biases account and initialise experiment tracking in Python
2. Log hyperparameters, evaluation metrics, and model configurations to W&B
3. Build comparative dashboards using parallel coordinates plots and run tables
4. Integrate W&B tracking with Optuna hyperparameter sweeps
5. Create shareable W&B Reports for stakeholder communication
6. Apply industry best practices for experiment naming, metric logging, and reproducibility

### Installation and Authentication

Before proceeding, ensure you have:
1. Created a free account at [wandb.ai](https://wandb.ai) (email or GitHub OAuth)
2. Installed the W&B library: `pip install wandb`
3. Authenticated via terminal: `wandb login` (paste API key from [wandb.ai/authorize](https://wandb.ai/authorize))

**Free Tier Includes:**
- Unlimited tracked hours and projects
- 100 GB cloud storage
- Up to 5 team member seats
- Full dashboard and report features

In [27]:
!pip install wandb

In [28]:
!wandb login

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\HARSH\_netrc.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


In [29]:
# Veryfy installation and check version
import wandb

print(f"W&B version: {wandb.__version__}")
print("\nAuthentication status: Logged in" if wandb.api.api_key else
"Authentication status: Not logged in - run 'wandb login' in terminal")

W&B version: 0.25.0

Authentication status: Logged in


In [30]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

# Generate synthetic real estate dataset
np.random.seed(42)
n = 200

sqft = np.random.normal(1500, 400, n)
bedrooms = np.random.randint(1, 6, n)
age = np.random.normal(20, 10, n).clip(1)
distance_to_city = np.random.exponential(5, n)
noise = np.random.normal(0, 20000, n)

# Price formula: realistic coefficients with added noise
price = (150 * sqft
         + 15000 * bedrooms
         - 1000 * age
         - 5000 * distance_to_city
         + noise)

df = pd.DataFrame({
    'sqft': sqft,
    'bedrooms': bedrooms,
    'age': age,
    'distance_to_city': distance_to_city,
    'price': price
})

X = df.drop('price', axis=1)
y = df['price']

print("Dataset shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())
print("\nSummary statistics:")
print(df.describe())

Dataset shape: (200, 5)

First 5 rows:
          sqft  bedrooms        age  distance_to_city          price
0  1698.685661         2  32.049229          1.642375  237528.113635
1  1444.694280         2   8.033395          7.337369  197399.581630
2  1759.075415         1  27.706783          1.035390  262254.857600
3  2109.211943         1  26.670561          1.955439  276125.759530
4  1406.338650         1  13.008534          2.770723  203781.689649

Summary statistics:
              sqft    bedrooms         age  distance_to_city          price
count   200.000000  200.000000  200.000000        200.000000     200.000000
mean   1483.691614    3.035000   20.603304          4.700595  222681.459890
std     372.401566    1.457531    9.836254          4.586707   69217.170502
min     452.101958    1.000000    1.000000          0.023214   59181.206580
25%    1217.948930    2.000000   14.543031          1.294452  176169.828129
50%    1498.323246    3.000000   20.121975          3.237268  221127.4

In [31]:
df

,sqft,bedrooms,age,distance_to_city,price
0,1698.685661,2,32.049229,1.642375,237528.113635
1,1444.694280,2,8.033395,7.337369,197399.581630
2,1759.075415,1,27.706783,1.035390,262254.857600
3,2109.211943,1,26.670561,1.955439,276125.759530
4,1406.338650,1,13.008534,2.770723,203781.689649
...,...,...,...,...,...
195,1654.126952,2,14.674197,0.099537,246275.981528
196,1146.457026,5,25.283166,0.549680,205146.179845
197,1561.490042,5,22.075035,8.045092,256822.121174
198,1523.283487,3,40.433205,0.983389,236507.426569


In [34]:
run = wandb.init(
    project = "Demo-1",
    name = "ridge-alpha-1.0",
    config = {
        "model":"Ridge",
        "alpha":1.0,
        "cv_folds":5,
        "scaler":"StandardScaler",
        "dataset_size":n

    }
)

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', Ridge(alpha=1.0))
])

# Evaluate using 5-fold cross-validation
scores = cross_val_score(pipe, X, y, cv=5, scoring='r2')

# Log metrics to W&B
wandb.log({
    'mean_r2': scores.mean(),
    'std_r2': scores.std(),
    'min_r2': scores.min(),
    'max_r2': scores.max()
})


print(f"Ridge (alpha=1.0): R² = {scores.mean():.4f} ± {scores.std():.4f}")
print(f"\nView this run at: {run.get_url()}")

# Close the run
wandb.finish()

wandb: WARNING The get_url method is deprecated and will be removed in a future release. Please use `run.url` instead.


Ridge (alpha=1.0): R² = 0.9031 ± 0.0070

View this run at: https://wandb.ai/harshvardhanaiml-institution/Demo-1/runs/ix3ru2is


max_r2,▁
mean_r2,▁
min_r2,▁
std_r2,▁
max_r2,0.91606
mean_r2,0.90311
min_r2,0.895
std_r2,0.00701


In [35]:
# Define models and hyperparameters to test
models = {
    'Ridge': Ridge,
    'Lasso': Lasso,
    'ElasticNet': ElasticNet
}

alphas = [0.01, 0.1, 1.0, 10.0]

print("="*60)
print("TRACKING 12 MODEL CONFIGURATIONS")
print("="*60)

for model_name, ModelClass in models.items():
    for alpha in alphas:
        # Initialize run with descriptive name and full configuration
        run = wandb.init(
            project="session-17-3-experiment-tracking",
            name=f"{model_name.lower()}-alpha-{alpha}",
            config={
                "model": model_name,
                "alpha": alpha,
                "cv_folds": 5,
                "scaler": "StandardScaler",
                "dataset_size": 200
            }
        )

        # Build and configure model
        if model_name == 'ElasticNet':
            model = ModelClass(alpha=alpha, l1_ratio=0.5)
        else:
            model = ModelClass(alpha=alpha)

        # Create pipeline and evaluate
        pipe = Pipeline([
            ('scaler', StandardScaler()),
            ('model', model)
        ])

        scores = cross_val_score(pipe, X, y, cv=5, scoring='r2')

        # Log comprehensive metrics
        wandb.log({
            'mean_r2': scores.mean(),
            'std_r2': scores.std(),
            'min_r2': scores.min(),
            'max_r2': scores.max(),
            'cv_range': scores.max() - scores.min()
        })

        print(f"{model_name:12} (α={alpha:>5}): R² = {scores.mean():.4f} ± {scores.std():.4f}")

        wandb.finish()

print("\n" + "="*60)
print("✓ All 12 runs logged to W&B!")
print("Open your W&B project dashboard to compare results.")
print("="*60)

TRACKING 12 MODEL CONFIGURATIONS


Ridge        (α= 0.01): R² = 0.9030 ± 0.0073


cv_range,▁
max_r2,▁
mean_r2,▁
min_r2,▁
std_r2,▁
cv_range,0.02164
max_r2,0.91682
mean_r2,0.90302
min_r2,0.89517
std_r2,0.00728


Ridge        (α=  0.1): R² = 0.9030 ± 0.0073


cv_range,▁
max_r2,▁
mean_r2,▁
min_r2,▁
std_r2,▁
cv_range,0.02159
max_r2,0.91675
mean_r2,0.90303
min_r2,0.89516
std_r2,0.00725


...

ElasticNet   (α=  1.0): R² = 0.8074 ± 0.0185


cv_range,▁
max_r2,▁
mean_r2,▁
min_r2,▁
std_r2,▁
cv_range,0.05108
max_r2,0.84353
mean_r2,0.80743
min_r2,0.79245
std_r2,0.0185


ElasticNet   (α= 10.0): R² = 0.2702 ± 0.0182


cv_range,▁
max_r2,▁
mean_r2,▁
min_r2,▁
std_r2,▁
cv_range,0.05086
max_r2,0.28651
mean_r2,0.27018
min_r2,0.23565
std_r2,0.01824



✓ All 12 runs logged to W&B!
Open your W&B project dashboard to compare results.


### Task: Add a 4th model type to the experiment loop: RandomForestRegressor from sklearn.ensemble.

- Import RandomForestRegressor
- Add it to the models dictionary with the key "RandomForest"
- Use n_estimators as the hyperparameter to vary (instead of alpha): - test [10, 50, 100, 200]
- Ensure the config dict includes "n_estimators" (not "alpha") for - - RandomForest runs
- Run the experiment loop

In [36]:
# 1. Putting them in Dict method
run = wandb.init(
    project="config-demo",
    config={"alpha": 0.1, "model": "Ridge"}
)

#2. Update after init(dynamic configs)
wandb.config.update({"learning_rate":0.01})

#3 Access Config values

alpha = wandb.config.alpha

print(f"Current alpha: {alpha}")
print(f"Full config: {dict(wandb.config)}")

wandb.finish()


Current alpha: 0.1
Full config: {'alpha': 0.1, 'model': 'Ridge', 'learning_rate': 0.01}


In [41]:
# W&B + Optuna Integration

import optuna
from optuna_integration.wandb import WeightsAndBiasesCallback

In [40]:
wandb_callback = WeightsAndBiasesCallback(
    metric_name="mean_r2",
    wandb_kwargs={
        "project": "optuna-sweep"
    }
)

def objective(trial):
    alpha = trial.suggest_float("alpha", 0.001, 10.0, log=True)
    l1_ratio = trial.suggest_float("l1_ratio", 0.0, 1.0)

    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('model', ElasticNet(alpha=alpha, l1_ratio=l1_ratio))
    ])

    scores = cross_val_score(pipe, X, y, cv=5, scoring='r2')
    return scores.mean()


study = optuna.create_study(direction="maximize")
study.optimize(objective,n_trials=30,callbacks=[wandb_callback])

print("="*60)
print("OPTUNA SWEEP COMPLETE")
print("="*60)
print(f"Best R²: {study.best_value:.4f}")
print(f"Best params: {study.best_params}")
print("\nAll 30 trials are now in W&B for visual analysis.")
print("="*60)

wandb.finish()

[I 2026-02-28 19:06:43,531] A new study created in memory with name: no-name-96eb8561-c29e-4d75-b6ac-5d02609824b1
[I 2026-02-28 19:06:43,578] Trial 0 finished with value: 0.9030692508385044 and parameters: {'alpha': 0.005224919075860147, 'l1_ratio': 0.45860852455633516}. Best is trial 0 with value: 0.9030692508385044.
[I 2026-02-28 19:06:43,608] Trial 1 finished with value: 0.7475169561911208 and parameters: {'alpha': 0.9625054616271144, 'l1_ratio': 0.24283587432566178}. Best is trial 0 with value: 0.9030692508385044.
[I 2026-02-28 19:06:43,645] Trial 2 finished with value: 0.9031218723746395 and parameters: {'alpha': 0.016132314409406438, 'l1_ratio': 0.05727952043163964}. Best is trial 2 with value: 0.9031218723746395.
[I 2026-02-28 19:06:43,675] Trial 3 finished with value: 0.8950223848079937 and parameters: {'alpha': 0.21618708448793428, 'l1_ratio': 0.4561271373861009}. Best is trial 2 with value: 0.9031218723746395.
[I 2026-02-28 19:06:43,701] Trial 4 finished with value: 0.9006040

OPTUNA SWEEP COMPLETE
Best R²: 0.9031
Best params: {'alpha': 0.03127696410515217, 'l1_ratio': 0.7036993074696369}

All 30 trials are now in W&B for visual analysis.


alpha,▁▂▁▁▁▁▁▁▁█▁▁▁▁▁▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁
l1_ratio,▄▃▁▄▂▁▄▁▁▅█▇▃▃▃▂▄▅▆▆▇▆▅▇▅▇▆▆█▄
mean_r2,█▆███████▁████████▇███████████
alpha,0.00269
l1_ratio,0.36581
mean_r2,0.90305
